In [1]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegressionCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

## 1. Data Preparation

In [2]:
track_history  = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/now_playing.csv")
track_analysis = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_analysis.csv")
track_metadata = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_metadata.csv")
weather        = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/weather.csv")

# convert all request times to datetime formats
for df in [track_history, track_analysis, track_metadata, weather]:
    df["request_time"] = pd.to_datetime(df["request_time"])

# converts a string in the format mm:ss to milliseconds
def mm_ss_to_ms(mm_ss_str):
    minutes, seconds = map(int, mm_ss_str.split(":"))
    return ((minutes * 60) + seconds) * 1000

# remove analysis/metadata observations whose id is null
track_analysis = track_analysis[~track_analysis["id"].isna()]
track_metadata = track_metadata[~track_metadata["id"].isna()]
# remove all metadata rows whose genres are null
track_metadata = track_metadata.dropna(subset=["genres"])
# if any values are missing in the track metadata (which is acceptable), set them to be an empty string
track_metadata.fillna("", inplace=True)
# change track analysis duration (mm:ss) to milliseconds
track_analysis["duration"] = track_analysis["duration"].apply(lambda d: mm_ss_to_ms(d))
# remove dB from track analysis loudness and convert to an int (e.g., "-2 dB" becomes -2)
track_analysis["loudness"] = track_analysis["loudness"].apply(lambda db: db.replace(" dB", "")).astype(np.int16)
# split camelot into three columns: X/y position via cos(camelot num * 2pi/12) and is_major
track_analysis["is_major"] = (track_analysis["camelot"].apply(lambda camelot: "B" in camelot)).astype(np.int16)
# turn camelot into coordinates (better for modeling; see explanation below)
track_analysis["camelot_x"] = np.cos(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)
track_analysis["camelot_y"] = np.sin(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)

# keep only relevant features
track_analysis = track_analysis[["request_id", "request_time", "mode", "camelot", "camelot_x", "camelot_y", "is_major", "duration", "tempo", "popularity", "energy", "happiness", "acousticness", "danceability", "instrumentalness", "speechiness", "liveness", "loudness"]]
track_history  = track_history[["request_id", "request_time", "title", "artist", "album", "duration", "program_start_ts", "plays", "is_ad"]]
track_metadata = track_metadata.drop(columns=["id", "artist_sort_name", "release_statuses", "release_languages", "release_scripts", "release_qualities"])

# separate ads and actual songs played into separate dataframes
track_ad_history = track_history[track_history["is_ad"]]
track_history    = track_history[~track_history["is_ad"]]

# one-hot encode genres, just in case.
genres = pd.concat([track_metadata[["request_id", "request_time"]], track_metadata["genres"].str.get_dummies(sep=" | ")], axis=1).drop("|", axis=1)

# combine tables
data = pd.merge(track_analysis, track_metadata, on=["request_id", "request_time"], how="inner")
data = pd.merge(data, track_history, on=["request_id", "request_time"], how="inner")
data = pd.merge(data, genres, on=["request_id", "request_time"], how="inner")

The camelot wheel is a circle, so standard linear numbers can confuse ML models. Specifically, a model might think 12 and 1 are far apart, even though they are neighbors. The solution used here is projecting the 12 positions onto a 2D coordinate system using cos/sin.

For example, 12 and 1 on the camelot wheel are in key (and should therefore have a small "distance"). 12 lies at point (1, 0); 1 at (.866, 0.5). Using euclidian distance, the distance between them is 0.51; the distance between 12 and 6 (opposite sides of the wheel) is 2.0, which we can use to describe that that 12 and 1 are very closely related, while 12 and 6 are mathematically distant (opposites on the camelot wheel).

## 2. Correlation Analysis

In [3]:
track_analysis_corr = track_analysis.corr(numeric_only=True)
track_analysis_corr_cols = track_analysis_corr.columns.to_list()

# ignore perfectly correlated values on the diagonal
# track_analysis_corr = track_analysis_corr.where(~np.eye(len(track_analysis_corr), dtype=bool))
# by itself, this returns duplicate pairs: track_analysis_corr.unstack().nlargest(5)... must first remove duplicates:
# track_analysis_corr.unstack()[track_analysis_corr.unstack().index.map(lambda x: x[0] < x[1])].nlargest(2), track_analysis_corr.unstack()[track_analysis_corr.unstack().index.map(lambda x: x[0] < x[1])].nsmallest(2)

alt.Chart(track_analysis_corr).mark_circle(opacity=0.5, size=25).encode(
    x=alt.X(alt.repeat("column"), type="quantitative"),
    y=alt.Y(alt.repeat("row"), type="quantitative")
).properties(width=150, height=150).repeat(row=track_analysis_corr_cols, column=track_analysis_corr_cols)

alt.RepeatChart(...)

## 3. Histograms / Misc Charts

In [4]:
most_played_artists = data.groupby("artist")[["artist"]].value_counts().sort_values(ascending=False).reset_index("artist").nlargest(50, columns=["count"])
most_popular_genres = genres.loc[:, "2-step":].sum().sort_values(ascending=False).nlargest(50).reset_index(name="count").rename({"index": "genre"}, axis=1)

most_played_artists_chart = alt.Chart(most_played_artists).encode(
    x=alt.X("artist", sort="-y", title="Artist name"),
    y=alt.Y("count:Q", title="Play count"),
    tooltip=["artist", "count"]
).mark_bar().properties(title="50 most played artists", width=600, height=300)

most_popular_genres_chart = alt.Chart(most_popular_genres).encode(
    x=alt.X("genre:N", sort="-y", title="Genre"),
    y=alt.Y("count", title="Count"),
    tooltip=["genre", "count"]
).mark_bar().properties(title="50 most played genres", width=600, height=300)

energy_distribution = alt.Chart(data).encode(
    x=alt.X("energy:O", bin=alt.Bin(maxbins=25), title="Energy"),
    y=alt.Y("count()", title="Count")
).mark_bar().properties(title="Energy distribution", width=600, height=300)

key_distribution = alt.Chart(data).encode(
    x=alt.X("camelot:O", title="Key"),
    y=alt.Y("count()", title="Count")
).mark_bar().properties(title="Key distribution", width=600, height=300)

popularity_distribution = alt.Chart(data).encode(
    x=alt.X("popularity:O", bin=alt.Bin(maxbins=25), title="Popularity"),
    y=alt.Y("count()", title="Count")
).mark_bar().properties(title="Popularity distribution", width=600, height=300)

happiness_distribution = alt.Chart(data).encode(
    x=alt.X("happiness:O", bin=alt.Bin(maxbins=25), title="Happiness"),
    y=alt.Y("count()", title="Count")
).mark_bar().properties(title="Happiness distribution", width=800, height=400)

most_played_artists_chart & most_popular_genres_chart & energy_distribution & key_distribution & popularity_distribution & happiness_distribution

alt.VConcatChart(...)

## 4. Song Recommendations

In [5]:
# song recommendations
songs = data.drop_duplicates(subset=["title_x", "artist_name"]).reset_index(drop=True)

# features = songs.loc[:, "camelot_x":"loudness"].drop(["camelot_x", "camelot_y", "is_major", "duration_x"], axis=1)
features = songs.select_dtypes(include='number')
# only apply standard scaler to features... want genres to hold more weight
model = make_pipeline(StandardScaler(), NearestNeighbors()).fit(features)

# # use PCA for dimensionality reduction so that we can "graph" songs
coords = make_pipeline(StandardScaler(), PCA(n_components=2)).fit_transform(features)
songs["x"], songs["y"] = coords[:, 0], coords[:, 1]

alt.Chart(songs).encode(
    x=alt.X("x:Q", title="", axis=alt.Axis(labels=False, ticks=False)),
    y=alt.Y("y:Q", title="", axis=alt.Axis(labels=False, ticks=False)),
    tooltip=alt.Tooltip(data.columns.to_list()),
    color=alt.Color("energy:Q", scale=alt.Scale(scheme="redyellowblue", reverse=True))
).mark_point(filled=True, size=100, color="green").properties(width=800, height=600, title="PCA Songs").interactive()

alt.Chart(...)

In [6]:
# find index of song to query (in this case, we query song 11 below)
print(songs[songs["artist"].str.lower() == "against all logic"][["artist", "title_x"]])

# 5 songs closest to song 10 in our list (must standardize song first)
def get_recs(song_index, n_recs=5):
    # add 1 because this will include the song itself
    _, rec_indices = model[-1].kneighbors(model[:-1].transform(features.iloc[[song_index]]), n_recs + 1)
    recs = songs.iloc[*rec_indices]
    # remove the query song first
    return recs

get_recs(11)[["artist", "title_x"]]

                artist                              title_x
11   Against All Logic                       Such a Bad Way
39   Against All Logic  You Are Going to Love Me and Scream
157  Against All Logic                    Some Kind of Game
170  Against All Logic                  Now U Got Me Hooked
201  Against All Logic                             Hopeless
223  Against All Logic                             Know You
270  Against All Logic                        I Never Dream
374  Against All Logic                     Flash in the Pan
381  Against All Logic         This Old House Is All I Have


,artist,title_x
11,Against All Logic,Such a Bad Way
29,TOKiMONSTA,Reverie
197,TOKiMONSTA,For You
151,Justice,Stress
381,Against All Logic,This Old House Is All I Have
47,Beach House,Space Song
